<a href="https://colab.research.google.com/github/Lobnaait/SEARCH_Lobna_Tsetline_CMRI/blob/main/Calibration_plot_Cetin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""Calibration (reliability) plots for the Cetin et al. 2017 SVM pipeline.

Place next to the classification script (methods/cetin2017/) and run it AFTER the classification
script, since it reads results/classification.json (selected features + kernel) and the radiomics CSVs.

SVC needs probability=True to output probabilities (Platt scaling, internal 5-fold CV).
Panels: training LOO (optimistic: features were selected on these same cases), test on `gt`
masks, test on `isensee` masks (if present).
"""
import json
import os

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

HERE = os.path.dirname(os.path.abspath(__file__))
N_BINS = 5  # small n (100 train / 50 test cases) -> few bins, otherwise they are almost empty


def svm_prob(kernel):
    return make_pipeline(StandardScaler(), SVC(kernel=kernel, C=1.0, probability=True, random_state=0))


def reliability(probs, y_true, classes, n_bins=N_BINS):
    """Top-label reliability: returns mean confidence, accuracy, counts per bin, ECE, Brier."""
    classes = np.asarray(classes)
    conf = probs.max(1)
    correct = (classes[probs.argmax(1)] == np.asarray(y_true)).astype(float)
    edges = np.linspace(1 / len(classes), 1.0, n_bins + 1)  # confidence >= 1/K for top label
    idx = np.clip(np.digitize(conf, edges) - 1, 0, n_bins - 1)
    mc, acc, cnt = np.full(n_bins, np.nan), np.full(n_bins, np.nan), np.zeros(n_bins, int)
    for b in range(n_bins):
        m = idx == b
        cnt[b] = m.sum()
        if m.any():
            mc[b], acc[b] = conf[m].mean(), correct[m].mean()
    ece = np.nansum(cnt / cnt.sum() * np.abs(acc - mc))
    onehot = (np.asarray(y_true)[:, None] == classes[None, :]).astype(float)
    brier = float(((probs - onehot) ** 2).sum(1).mean())
    return mc, acc, cnt, float(ece), brier


def main():
    res = json.load(open(os.path.join(HERE, 'results', 'classification.json')))
    kernel = res['kernel']
    sel_names = res['kernels'][kernel]['selected']

    gt = pd.read_csv(os.path.join(HERE, 'results', 'radiomics_gt.csv'))
    tr, te = gt[gt.pid <= 100], gt[gt.pid > 100]
    Xtr, ytr = tr[sel_names].values, tr.group.values
    sets = {}

    # 1) training LOO probabilities
    loo_p = cross_val_predict(svm_prob(kernel), Xtr, ytr, cv=LeaveOneOut(), method='predict_proba')
    classes = np.unique(ytr)
    sets['Train LOO (optimistic)'] = (loo_p, ytr)

    # 2) test sets with the final model
    model = svm_prob(kernel).fit(Xtr, ytr)
    sets['Test - GT masks'] = (model.predict_proba(te[sel_names].values), te.group.values)
    auto = os.path.join(HERE, 'results', 'radiomics_isensee.csv')
    if os.path.exists(auto):
        a = pd.read_csv(auto)
        for c in sel_names:
            if c not in a:
                a[c] = 0.0
        sets['Test - Isensee masks'] = (model.predict_proba(a[sel_names].values), a.group.values)

    fig, axes = plt.subplots(2, len(sets), figsize=(4.6 * len(sets), 7), squeeze=False,
                             gridspec_kw={'height_ratios': [3, 1]})
    for j, (title, (p, y)) in enumerate(sets.items()):
        mc, acc, cnt, ece, brier = reliability(p, y, classes)
        ax = axes[0, j]
        ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
        ax.plot(mc, acc, 'o-', label=f'SVM-{kernel} (ECE={ece:.3f})')
        for x_, y_, n_ in zip(mc, acc, cnt):
            if n_:
                ax.annotate(str(n_), (x_, y_), textcoords='offset points', xytext=(4, -10), fontsize=8)
        ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='Mean predicted confidence', ylabel='Accuracy',
               title=f'{title}\nBrier={brier:.3f}, n={len(y)}')
        ax.legend(loc='upper left')
        axes[1, j].hist(p.max(1), bins=np.linspace(1 / len(classes), 1, N_BINS + 1), edgecolor='k')
        axes[1, j].set(xlabel='Confidence', ylabel='Count')
    fig.suptitle('Cetin et al. 2017 (SVM + SFS) - calibration', y=1.0)
    fig.tight_layout()
    out = os.path.join(HERE, 'results', 'calibration_plot.png')
    fig.savefig(out, dpi=200, bbox_inches='tight')
    print('saved', out)


if __name__ == '__main__':
    main()